# PLAN classification: final test results and evaluation report

This notebook loads the frozen pipelines and the test split only. It reports every requested evaluation metric, compares against both baselines, and then provides descriptive feature, error, limitation, and model-summary sections.

Test results are descriptive because an earlier notebook version inspected this holdout. No test result is used to change the model or threshold.


In [5]:
import json
from pathlib import Path
import duckdb
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, precision_score, recall_score, roc_auc_score, roc_curve
SEED = 20261021
BOOTSTRAPS = 500
MAX_FALSE_ALARM_RATE = 0.10
TARGET = "reenters_within_horizon"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/processed_plan/panel.csv").is_file())
DATA_DIR = ROOT / "data/processed_plan"
ARTIFACT_DIR = DATA_DIR / "model_artifacts"
manifest_path = ARTIFACT_DIR / "manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError("Run the training/validation notebook first to create frozen model artifacts.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest["target"] == TARGET
artifact_sklearn = manifest.get("scikit_learn_version")
if artifact_sklearn is None:
    raise RuntimeError("This model manifest has no scikit-learn version. Rerun the training/validation notebook first.")
if sklearn.__version__ != artifact_sklearn:
    raise RuntimeError(
        f"Model artifacts use scikit-learn {artifact_sklearn}, but this kernel has {sklearn.__version__}. "
        "Select a kernel with the matching version, or rerun the training/validation notebook in this kernel first."
    )
model_specs = manifest["models"]
NEEDED = set(["split", "norad_cat_id", "obs_date", TARGET, "f107_adj_mean_81d",
              "perigee_km_asof", "perigee_decay_rate_km_per_day", "object_type", "rcs_size"])
for spec in model_specs.values():
    NEEDED.update(spec["features"])
quoted = ", ".join('"' + column.replace('"', '""') + '"' for column in sorted(NEEDED))
# This query materializes test records only; the notebook never reads train or validation rows.
test = duckdb.connect().execute(
    f"SELECT {quoted} FROM read_csv_auto(?) WHERE split = 'test'",
    [str(DATA_DIR / "panel.csv")],
).df()
test["obs_date"] = pd.to_datetime(test["obs_date"])
assert not test.empty and test["split"].eq("test").all()
assert TARGET in test.columns
print(f"Loaded test split only: {len(test):,} rows; {test.norad_cat_id.nunique():,} objects.")
print("Caution: an earlier notebook version already inspected this holdout, so these results are descriptive, not an untouched final test.")
def recall_at_fpr(y, probability, sample_weight=None):
    fpr, tpr, _ = roc_curve(y, probability, sample_weight=sample_weight)
    allowed = fpr <= MAX_FALSE_ALARM_RATE
    return float(tpr[allowed].max()) if allowed.any() else 0.0
def metric_values(y, probability, sample_weight=None):
    probability = np.asarray(probability, dtype=float)
    prediction = probability >= 0.5
    return {
        "Accuracy (threshold 0.50)": float(accuracy_score(y, prediction, sample_weight=sample_weight)),
        "Precision (threshold 0.50)": float(precision_score(y, prediction, sample_weight=sample_weight, zero_division=0)),
        "Recall (threshold 0.50)": float(recall_score(y, prediction, sample_weight=sample_weight, zero_division=0)),
        "ROC AUC": float(roc_auc_score(y, probability, sample_weight=sample_weight)),
        "PR-AUC": float(average_precision_score(y, probability, sample_weight=sample_weight)),
        "Brier score": float(brier_score_loss(y, probability, sample_weight=sample_weight)),
        "Recall at <=10% false-alarm rate": recall_at_fpr(y, probability, sample_weight),
    }
def object_bootstrap(y, probability, groups, seed):
    y = np.asarray(y, dtype=np.int8)
    probability = np.asarray(probability, dtype=float)
    codes, object_ids = pd.factorize(np.asarray(groups), sort=True)
    if len(object_ids) < 2:
        raise ValueError("Object bootstrap requires at least two objects.")
    rng = np.random.default_rng(seed)
    samples = {name: [] for name in metric_values(y, probability)}
    n_objects = len(object_ids)
    for _ in range(BOOTSTRAPS):
        draws = rng.integers(0, n_objects, size=n_objects)
        multiplicity = np.bincount(draws, minlength=n_objects)
        values = metric_values(y, probability, sample_weight=multiplicity[codes])
        for name, value in values.items():
            samples[name].append(value)
    point = metric_values(y, probability)
    return {
        name: {"estimate": point[name], "low": float(np.quantile(values, .025)),
               "high": float(np.quantile(values, .975))}
        for name, values in samples.items()
    }
def threshold_operating_metrics(y, probability, threshold):
    prediction = np.asarray(probability) >= float(threshold)
    y = np.asarray(y, dtype=bool)
    negatives = ~y
    return {
        "Validation-locked threshold": float(threshold),
        "Accuracy at validation-locked threshold": float(accuracy_score(y, prediction)),
        "Precision at validation-locked threshold": float(precision_score(y, prediction, zero_division=0)),
        "Recall at validation-locked threshold": float(recall_score(y, prediction, zero_division=0)),
        "False-alarm rate at validation-locked threshold": float(prediction[negatives].mean()) if negatives.any() else float("nan"),
    }
def baseline_physics(frame):
    perigee = pd.to_numeric(frame["perigee_km_asof"], errors="coerce").to_numpy()
    rate = pd.to_numeric(frame["perigee_decay_rate_km_per_day"], errors="coerce").to_numpy()
    days_to_120 = np.divide(perigee - 120.0, -rate, out=np.full(len(frame), np.inf), where=rate < 0)
    return ((perigee <= 120.0) | ((rate < 0) & (days_to_120 <= 365.0))).astype(float)
probabilities, summaries, thresholds = {}, {}, {}
for i, (name, spec) in enumerate(model_specs.items(), start=1):
    model = joblib.load(ARTIFACT_DIR / spec["artifact"])
    probability = model.predict_proba(test[spec["features"]])[:, 1]
    probabilities[name] = probability
    summaries[name] = object_bootstrap(test[TARGET], probability, test["norad_cat_id"], SEED + 1000 + i)
    thresholds[name] = spec["validation_threshold_at_10pct_fpr"]
    operating = threshold_operating_metrics(test[TARGET], probability, thresholds[name])
    print(f"\n{name}")
    for metric, result in summaries[name].items():
        print(f"  {metric}: {result['estimate']:.4f} [95% object bootstrap CI {result['low']:.4f}, {result['high']:.4f}]")
    print("  Test operating metrics at validation-locked threshold:")
    for key, value in operating.items():
        print(f"    {key}: {value:.4f}")
print("\nTest baselines:")
baseline_probabilities = {
    "Training-rate baseline": np.full(len(test), manifest["training_rate"], dtype=float),
    "Perigee-decay physics baseline": baseline_physics(test),
}
for i, (name, probability) in enumerate(baseline_probabilities.items(), start=1):
    result = object_bootstrap(test[TARGET], probability, test["norad_cat_id"], SEED + 1100 + i)
    print(name)
    for metric, record in result.items():
        print(f"  {metric}: {record['estimate']:.4f} [95% object bootstrap CI {record['low']:.4f}, {record['high']:.4f}]")
def paired_delta_bootstrap(y, first, second, groups, seed):
    y = np.asarray(y, dtype=np.int8)
    codes, objects = pd.factorize(np.asarray(groups), sort=True)
    rng = np.random.default_rng(seed)
    deltas = {m: [] for m in metric_values(y, first)}
    for _ in range(BOOTSTRAPS):
        mult = np.bincount(rng.integers(0, len(objects), len(objects)), minlength=len(objects))
        a = metric_values(y, first, mult[codes])
        b = metric_values(y, second, mult[codes])
        for metric in deltas:
            deltas[metric].append(b[metric] - a[metric])
    return {m: (float(np.quantile(v, .025)), float(np.quantile(v, .975))) for m, v in deltas.items()}
print("\nTest Sun minus no-Sun paired object-bootstrap deltas (descriptive only):")
for algorithm, no_sun, with_sun in [
    ("Logistic regression", "Logistic regression without solar", "Logistic regression with solar"),
    ("Histogram gradient boosting", "Histogram gradient boosting without solar", "Histogram gradient boosting with solar"),
]:
    delta = paired_delta_bootstrap(test[TARGET], probabilities[no_sun], probabilities[with_sun],
                                   test["norad_cat_id"], SEED + 1200)
    point_a = metric_values(test[TARGET], probabilities[no_sun])
    point_b = metric_values(test[TARGET], probabilities[with_sun])
    print(algorithm)
    for metric, ci in delta.items():
        print(f"  {metric} change (with Sun minus without Sun): {point_b[metric]-point_a[metric]:+.4f} [95% paired object bootstrap CI: {ci[0]:+.4f} to {ci[1]:+.4f}]")
cut1, cut2 = manifest["solar_band_cutpoints_f107_adj_mean_81d"]
solar_bins = [-np.inf, cut1, cut2, np.inf]
solar_labels = ["low", "middle", "high"]
solar_values = pd.to_numeric(test["f107_adj_mean_81d"], errors="coerce")
solar_band = pd.cut(solar_values, solar_bins, labels=solar_labels, include_lowest=True)
print(f"\nTest F10.7 trailing 81-day mean: n={solar_values.notna().sum():,}; min={solar_values.min():.2f}; "
      f"median={solar_values.median():.2f}; max={solar_values.max():.2f}")
print("Test objects by train-defined solar band: " + str(
    test.assign(_band=solar_band).groupby("_band", observed=False)["norad_cat_id"].nunique().reindex(solar_labels, fill_value=0).to_dict()
))
selected = manifest["selected_model"]
selected_spec = model_specs[selected]
selected_prediction = probabilities[selected] >= thresholds[selected]
error_data = test[["norad_cat_id", TARGET, "perigee_km_asof", "object_type", "rcs_size"]].copy()
error_data["prediction"] = selected_prediction.astype("int8")
error_data["error"] = error_data["prediction"].ne(error_data[TARGET].astype("int8"))
error_data["error_type"] = np.select(
    [(error_data.prediction == 1) & (error_data[TARGET] == 0),
     (error_data.prediction == 0) & (error_data[TARGET] == 1)],
    ["false_positive", "false_negative"], default="correct",
)
error_data["perigee_band"] = pd.cut(error_data["perigee_km_asof"],
    bins=[-np.inf, 200, 400, 600], labels=["<=200 km", "200-400 km", "400-600 km"])
error_data["solar_band"] = solar_band.to_numpy()
print(f"\nPost-test error analysis for validation-selected model: {selected}.")
print("Test is used here only for the preplanned descriptive error analysis; do not revise model choices from it.")
for group in ["perigee_band", "object_type", "rcs_size", "solar_band"]:
    table = error_data.groupby(group, observed=True, dropna=False).agg(
        rows=(TARGET, "size"), objects=("norad_cat_id", "nunique"),
        positive_rate=(TARGET, "mean"), error_rate=("error", "mean"),
        false_positives=("error_type", lambda v: int((v == "false_positive").sum())),
        false_negatives=("error_type", lambda v: int((v == "false_negative").sum())),
    ).reset_index()
    print(f"\nBy {group}:")
    print(table.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

print("\n=== Complete test evaluation metrics (descriptive post-test report) ===")
test_rows = []
for name, result in summaries.items():
    spec = model_specs[name]
    row = {"Model": name, "Algorithm": spec["algorithm"], "Features": len(spec["features"]),
           "Threshold": thresholds[name]}
    row.update({metric: record["estimate"] for metric, record in result.items()})
    row.update({f"{metric} CI low": record["low"] for metric, record in result.items()})
    row.update({f"{metric} CI high": record["high"] for metric, record in result.items()})
    test_rows.append(row)
for name, probability in baseline_probabilities.items():
    result = object_bootstrap(test[TARGET], probability, test["norad_cat_id"], SEED + 1300 + len(test_rows))
    row = {"Model": name, "Algorithm": "baseline", "Features": 0, "Threshold": np.nan}
    row.update({metric: record["estimate"] for metric, record in result.items()})
    row.update({f"{metric} CI low": record["low"] for metric, record in result.items()})
    row.update({f"{metric} CI high": record["high"] for metric, record in result.items()})
    test_rows.append(row)
all_metrics_table = pd.DataFrame(test_rows)
metric_columns = [
    "Model", "Algorithm", "Features", "Threshold",
    "Accuracy (threshold 0.50)", "Precision (threshold 0.50)",
    "Recall (threshold 0.50)", "ROC AUC", "PR-AUC", "Brier score",
    "Recall at <=10% false-alarm rate",
]
print(all_metrics_table[metric_columns].to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("\n95% object-bootstrap intervals:")
ci_columns = ["Model"] + [column for column in all_metrics_table.columns if column.endswith("CI low") or column.endswith("CI high")]
print(all_metrics_table[ci_columns].to_string(index=False, float_format=lambda value: f"{value:.4f}"))

print("\n=== Test operating points at validation-locked thresholds ===")
operating_rows = []
for name, probability in probabilities.items():
    operating_rows.append({"Model": name, **threshold_operating_metrics(test[TARGET], probability, thresholds[name])})
print(pd.DataFrame(operating_rows).to_string(index=False, float_format=lambda value: f"{value:.4f}"))

print("\n=== Feature analysis and recommendations (descriptive; no post-test model changes) ===")
selected_model = joblib.load(ARTIFACT_DIR / selected_spec["artifact"])
selected_features = selected_spec["features"]
selected_input = test[selected_features]
selected_base = selected_model.predict_proba(selected_input)[:, 1]
selected_ap = average_precision_score(test[TARGET], selected_base)
selected_brier = brier_score_loss(test[TARGET], selected_base)
feature_rows = []
feature_rng = np.random.default_rng(SEED + 1400)
for feature in selected_features:
    ap_drops, brier_increases = [], []
    for _ in range(5):
        shuffled = selected_input.copy()
        shuffled[feature] = selected_input.iloc[feature_rng.permutation(len(selected_input))][feature].to_numpy()
        probability = selected_model.predict_proba(shuffled)[:, 1]
        ap_drops.append(selected_ap - average_precision_score(test[TARGET], probability))
        brier_increases.append(brier_score_loss(test[TARGET], probability) - selected_brier)
    feature_rows.append({"Feature": feature, "AP drop": np.mean(ap_drops), "Brier increase": np.mean(brier_increases)})
feature_table = pd.DataFrame(feature_rows).sort_values(["AP drop", "Brier increase"], ascending=False)
print("Top individual inputs by permutation impact:")
print(feature_table.head(15).to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("Recommendations:")
if feature_table.empty:
    print("  No feature recommendations can be made because the selected pipeline has no inputs.")
else:
    top_features = ", ".join(feature_table.head(5)["Feature"].tolist())
    print(f"  Preserve and monitor the highest-impact inputs: {top_features}.")
    print("  Keep solar-weather inputs as an ablation decision, not a causal claim; validate any change on a new split.")
    print("  Investigate high-impact correlated orbit variables together before removing or interpreting one feature alone.")
print("  Permutation impact is predictive association, not causal importance, and was computed after the test evaluation.")

print("\n=== Post-test error analysis for the validation-selected model ===")
print("Counts are included for every subgroup; use these results descriptively and do not retune the frozen model.")
for group in ["perigee_band", "object_type", "rcs_size", "solar_band"]:
    table = error_data.groupby(group, observed=True, dropna=False).agg(
        rows=(TARGET, "size"), objects=("norad_cat_id", "nunique"),
        positive_rate=(TARGET, "mean"), error_rate=("error", "mean"),
        false_positives=("error_type", lambda values: int((values == "false_positive").sum())),
        false_negatives=("error_type", lambda values: int((values == "false_negative").sum())),
    ).reset_index()
    print(f"\nBy {group}:")
    print(table.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

print("\n=== Models summary ===")
model_summary = []
for name, spec in model_specs.items():
    model_summary.append({
        "Model": name, "Algorithm": spec["algorithm"], "Solar": "with trailing solar" if "with solar" in name else "without solar",
        "Input count": len(spec["features"]), "Validation PR-AUC": spec["validation_metrics"]["PR-AUC"],
        "Test PR-AUC": summaries[name]["PR-AUC"]["estimate"], "Test Brier": summaries[name]["Brier score"]["estimate"],
        "Locked threshold": thresholds[name], "Selected": name == selected,
    })
print(pd.DataFrame(model_summary).to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print(f"Selection rule: {manifest['decision_rule']}; selected model: {selected}.")

print("\n=== FINAL TEST RESULTS: ALL MODELS AND BASELINES ===")
clean_test_columns = {
    "Accuracy (threshold 0.50)": "Accuracy",
    "Precision (threshold 0.50)": "Precision",
    "Recall (threshold 0.50)": "Recall",
    "ROC AUC": "ROC AUC",
    "PR-AUC": "PR-AUC",
    "Brier score": "Brier",
    "Recall at <=10% false-alarm rate": "Recall at 10% FPR",
}
clean_test_table = all_metrics_table[["Model", "Algorithm"] + list(clean_test_columns)].rename(columns=clean_test_columns)
print(clean_test_table.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("\n=== FINAL TEST OPERATING POINTS ===")
print(pd.DataFrame(operating_rows).to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("\nMetric guide: higher is better for Accuracy, Precision, Recall, ROC AUC, PR-AUC, and recall at 10% FPR; lower is better for Brier score.")


Loaded test split only: 6,921 rows; 508 objects.
Caution: an earlier notebook version already inspected this holdout, so these results are descriptive, not an untouched final test.

Logistic regression without solar
  Accuracy (threshold 0.50): 0.8853 [95% object bootstrap CI 0.8664, 0.9047]
  Precision (threshold 0.50): 0.7217 [95% object bootstrap CI 0.6526, 0.7860]
  Recall (threshold 0.50): 0.7357 [95% object bootstrap CI 0.6979, 0.7741]
  ROC AUC: 0.9368 [95% object bootstrap CI 0.9224, 0.9505]
  PR-AUC: 0.8292 [95% object bootstrap CI 0.7964, 0.8609]
  Brier score: 0.0830 [95% object bootstrap CI 0.0719, 0.0940]
  Recall at <=10% false-alarm rate: 0.7750 [95% object bootstrap CI 0.7238, 0.8292]
  Test operating metrics at validation-locked threshold:
    Validation-locked threshold: 0.5096
    Accuracy at validation-locked threshold: 0.8869
    Precision at validation-locked threshold: 0.7293
    Recall at validation-locked threshold: 0.7308
    False-alarm rate at validation-loc

## Interpretation, feature analysis, and error analysis

Read the final test table first. The feature analysis is descriptive permutation analysis, not causal importance. Error tables always include row and object counts so small groups are visible.
